In [ ]:
# Portable project paths. Set TLS_PROJECT_ROOT to the directory containing the input data.
import os
from pathlib import Path
PROJECT_ROOT = Path(os.environ.get("TLS_PROJECT_ROOT", ".")).resolve()


# Supplementary Figure 34 plotting code


## Shared setup


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon, Rectangle, Circle, FancyArrowPatch
from scipy.spatial import ConvexHull, QhullError

ROOT = Path.cwd()
DATA = ROOT / "source_data"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)
assert DATA.exists(), "Run this notebook from the Figure 6 code directory."

mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "font.family": "DejaVu Sans",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "legend.frameon": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

blue = "#2B5C96"
red = "#C94C4C"
grey = "#8B9AA7"
dark = "#152536"

cell_colors = {
    "Epithelial": "#1F7A7A", "Fibroblast": "#E59B3A", "Endothelial": "#39A56A",
    "Pericyte": "#93B5D7", "T cell": "#E64B35", "NK cell": "#7E57C2",
    "B cell": "#2C7FB8", "Plasma cell": "#4EA3D8", "Monocyte/Macrophage": "#61C2A2",
    "Dendritic cell": "#D94F9B", "Mast cell": "#8CD36B", "Neutrophil": "#37BDBD",
    "Mix": "#C9C9C9", "Unknown": "#D9D9D9",
    "T/NK cell": "#4FA366", "Myeloid": "#EF6C42", "Fibroblast/FDC": "#F1A55C",
    "Endothelial/Pericyte": "#5EB4A5", "Other/Unknown": "#BDBDBD", "Other immune": "#BDBDBD",
    "B cells": "#2C7FB8", "DC": "#D94F9B", "DC cells": "#D94F9B",
    "Endothelial cells": "#39A56A", "Epithelial cells": "#1F7A7A",
    "Fibroblasts": "#E59B3A", "Macrophages": "#61C2A2",
    "Mast cells": "#8CD36B", "Monocytes": "#61C2A2",
    "T cells": "#E64B35", "NK cells": "#7E57C2"
}

class_colors = {"Conforming TLS": blue, "Deviating TLS": red, "Mature TLS": "#2B7A3D"}

def savefig(fig, name):
    for ext in ["pdf", "svg", "png"]:
        fig.savefig(OUT / f"{name}.{ext}", dpi=450, bbox_inches="tight")

def sem(x):
    x = pd.Series(x).dropna()
    return x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan

def draw_hull(ax, df, x="X", y="Y", color="k", lw=0.55, alpha=1, pad=0):
    pts = df[[x, y]].dropna().drop_duplicates().to_numpy()
    if len(pts) < 3:
        return
    try:
        h = ConvexHull(pts)
        poly = Polygon(pts[h.vertices], closed=True, fill=False, ec=color, lw=lw, alpha=alpha, joinstyle="round")
        ax.add_patch(poly)
    except QhullError:
        pass

def add_panel(ax, label):
    ax.text(-0.08, 1.05, label, transform=ax.transAxes, ha="left", va="bottom", fontsize=9, fontweight="bold")
